# World Cup Predictive Modeling (2002–2022)
Full pipeline: Data Acquisition → Feature Engineering → Model Training → Prediction

## 0. Install & Import Dependencies

In [ ]:
# Run this cell once to install all required packages
!pip install xgboost lightgbm pandas numpy scikit-learn requests scipy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import requests
import math
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from scipy.stats import poisson
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, classification_report
import xgboost as xgb
from xgboost import XGBRegressor
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

print('All imports successful.')

## 1. Data Sourcing

### 1A. Historical Match Data — OpenFootball (2002–2022 only)

In [ ]:
# Scoped strictly to 2002–2022: modern era with reliable stats and WB economic data

AVAILABLE_YEARS = [2002, 2006, 2010, 2014, 2018, 2022]
YEAR_MIN, YEAR_MAX = 2002, 2022

BASE_URL = 'https://raw.githubusercontent.com/openfootball/world-cup.json/master/{year}/worldcup.json'

def fetch_wc_matches(year):
    """Fetch and flatten all matches for a given World Cup year."""
    url = BASE_URL.format(year=year)
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f'  Could not fetch {year}: {e}')
        return []

    rows = []
    for round_data in data.get('rounds', []):
        stage = round_data.get('name', '')
        is_knockout = int(any(kw in stage.lower() for kw in
                              ['final', 'semi', 'quarter', 'round of', 'knockout', 'third']))
        for m in round_data.get('matches', []):
            score = m.get('score', {})
            ft = score.get('ft', [None, None])
            rows.append({
                'year'       : year,
                'date'       : m.get('date'),
                'stage'      : stage,
                'is_knockout': is_knockout,
                'team_a'     : m['team1'].get('name') if 'team1' in m else None,
                'team_b'     : m['team2'].get('name') if 'team2' in m else None,
                'goals_a'    : ft[0] if ft[0] is not None else np.nan,
                'goals_b'    : ft[1] if ft[1] is not None else np.nan,
            })
    return rows

all_matches = []
for yr in AVAILABLE_YEARS:
    print(f'Fetching {yr}...')
    all_matches.extend(fetch_wc_matches(yr))

df_matches = pd.DataFrame(all_matches)
df_matches['date'] = pd.to_datetime(df_matches['date'], errors='coerce')
df_matches.dropna(subset=['goals_a', 'goals_b', 'team_a', 'team_b'], inplace=True)
df_matches['goals_a'] = df_matches['goals_a'].astype(int)
df_matches['goals_b'] = df_matches['goals_b'].astype(int)

# Hard filter: keep only 2002–2022
df_matches = df_matches[
    (df_matches['year'] >= YEAR_MIN) & (df_matches['year'] <= YEAR_MAX)
].reset_index(drop=True)

def get_outcome(row):
    if row['goals_a'] > row['goals_b']:
        return 0  # Team A win
    elif row['goals_a'] == row['goals_b']:
        return 1  # Draw
    else:
        return 2  # Team B win

df_matches['outcome'] = df_matches.apply(get_outcome, axis=1)
print(f'Years covered : {sorted(df_matches["year"].unique())}')
print(f'Total matches : {len(df_matches)}')
df_matches.head()

### 1B. Elo Rating System (Dynamic, Match-by-Match)

In [ ]:
ELO_START   = 1500
K_WORLD_CUP = 60

elo_ratings = {}  # team_name -> current elo

def get_elo(team):
    return elo_ratings.get(team, ELO_START)

def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(elo_a, elo_b, result_a, k=K_WORLD_CUP):
    """result_a: 1=win, 0.5=draw, 0=loss for Team A"""
    exp_a = expected_score(elo_a, elo_b)
    new_a = elo_a + k * (result_a - exp_a)
    new_b = elo_b + k * ((1 - result_a) - (1 - exp_a))
    return new_a, new_b

# Sort chronologically — compute pre-match Elo snapshot, then update
df_matches_sorted = df_matches.sort_values('date').reset_index(drop=True)

elo_a_list, elo_b_list = [], []

for _, row in df_matches_sorted.iterrows():
    ta, tb = row['team_a'], row['team_b']
    ea, eb = get_elo(ta), get_elo(tb)
    elo_a_list.append(ea)
    elo_b_list.append(eb)

    if row['goals_a'] > row['goals_b']:
        result_a = 1.0
    elif row['goals_a'] == row['goals_b']:
        result_a = 0.5
    else:
        result_a = 0.0

    new_ea, new_eb = update_elo(ea, eb, result_a)
    elo_ratings[ta] = new_ea
    elo_ratings[tb] = new_eb

df_matches_sorted['elo_a']   = elo_a_list
df_matches_sorted['elo_b']   = elo_b_list
df_matches_sorted['elo_diff'] = df_matches_sorted['elo_a'] - df_matches_sorted['elo_b']

print('Elo ratings computed.')
df_matches_sorted[['date', 'team_a', 'team_b', 'elo_a', 'elo_b', 'elo_diff']].head()

### 1C. Socioeconomic Data — World Bank API (year-accurate, 2002–2022)

In [ ]:
# World Bank API — GDP per capita (PPP) and Population fetched for the exact match year.
# PPP data is reliable from ~1990 onward, so the 2002–2022 window is fully covered.
# Indicator codes:
#   NY.GDP.PCAP.PP.CD = GDP per capita, PPP (current international $)
#   SP.POP.TOTL       = Total population

WB_BASE = 'https://api.worldbank.org/v2/country/{iso}/indicator/{indicator}?date={year}&format=json&per_page=1'

# Team name → ISO3 mapping (covers all 2002–2022 WC participants)
TEAM_TO_ISO3 = {
    'Brazil': 'BRA', 'Germany': 'DEU', 'Italy': 'ITA', 'Argentina': 'ARG',
    'France': 'FRA', 'England': 'GBR', 'Spain': 'ESP', 'Netherlands': 'NLD',
    'Uruguay': 'URY', 'Croatia': 'HRV', 'Belgium': 'BEL', 'Portugal': 'PRT',
    'Mexico': 'MEX', 'Japan': 'JPN', 'South Korea': 'KOR', 'USA': 'USA',
    'Australia': 'AUS', 'Switzerland': 'CHE', 'Denmark': 'DNK', 'Sweden': 'SWE',
    'Poland': 'POL', 'Senegal': 'SEN', 'Morocco': 'MAR', 'Ghana': 'GHA',
    'Nigeria': 'NGA', 'Cameroon': 'CMR', 'Saudi Arabia': 'SAU', 'Iran': 'IRN',
    'Ecuador': 'ECU', 'Qatar': 'QAT', 'Canada': 'CAN', 'Serbia': 'SRB',
    'Wales': 'GBR', 'Czech Republic': 'CZE', 'Turkey': 'TUR', 'Russia': 'RUS',
    'Colombia': 'COL', 'Chile': 'CHL', 'Costa Rica': 'CRI', 'Honduras': 'HND',
    'Ivory Coast': 'CIV', 'Tunisia': 'TUN', 'Algeria': 'DZA', 'Egypt': 'EGY',
    'South Africa': 'ZAF', 'Greece': 'GRC', 'Slovakia': 'SVK', 'Slovenia': 'SVN',
    'Austria': 'AUT', 'Ukraine': 'UKR', 'New Zealand': 'NZL', 'Paraguay': 'PRY',
    'Trinidad and Tobago': 'TTO', 'Angola': 'AGO', 'Togo': 'TGO', 'Iraq': 'IRQ',
    'North Korea': 'PRK', 'Slovakia': 'SVK', 'Bosnia and Herzegovina': 'BIH',
}

_wb_cache = {}

def fetch_wb_indicator(iso3, indicator, year):
    """Fetch a single World Bank indicator; caches results to avoid duplicate calls."""
    key = (iso3, indicator, year)
    if key in _wb_cache:
        return _wb_cache[key]
    url = WB_BASE.format(iso=iso3, indicator=indicator, year=year)
    try:
        r = requests.get(url, timeout=8)
        payload = r.json()
        value = payload[1][0]['value'] if payload and len(payload) > 1 and payload[1] else None
        _wb_cache[key] = value
        return value
    except:
        _wb_cache[key] = None
        return None

def get_socioeconomic_diff(team_a, team_b, year):
    """Returns (log_gdp_diff, log_pop_diff) for Team A minus Team B at the given year."""
    iso_a = TEAM_TO_ISO3.get(team_a)
    iso_b = TEAM_TO_ISO3.get(team_b)
    if not iso_a or not iso_b:
        return np.nan, np.nan

    gdp_a = fetch_wb_indicator(iso_a, 'NY.GDP.PCAP.PP.CD', year)
    gdp_b = fetch_wb_indicator(iso_b, 'NY.GDP.PCAP.PP.CD', year)
    pop_a = fetch_wb_indicator(iso_a, 'SP.POP.TOTL', year)
    pop_b = fetch_wb_indicator(iso_b, 'SP.POP.TOTL', year)

    log_gdp_diff = (math.log(gdp_a) - math.log(gdp_b)) if gdp_a and gdp_b else np.nan
    log_pop_diff = (math.log(pop_a) - math.log(pop_b)) if pop_a and pop_b else np.nan
    return log_gdp_diff, log_pop_diff

print('World Bank functions ready.')
print('Each match fetches GDP + Population for both teams at the exact tournament year.')

### 1D. Attach Socioeconomic Differentials to Match Dataset
Each match pulls World Bank data for its **exact tournament year** (2002, 2006, 2010, 2014, 2018, or 2022).

In [ ]:
log_gdp_diffs, log_pop_diffs = [], []
total = len(df_matches_sorted)

for i, row in df_matches_sorted.iterrows():
    if i % 50 == 0:
        print(f'  Processing match {i + 1}/{total} (year={row["year"]})...')
    gdp_diff, pop_diff = get_socioeconomic_diff(
        row['team_a'], row['team_b'], int(row['year'])
    )
    log_gdp_diffs.append(gdp_diff)
    log_pop_diffs.append(pop_diff)

df_matches_sorted['log_gdp_diff'] = log_gdp_diffs
df_matches_sorted['log_pop_diff'] = log_pop_diffs

gdp_ok = df_matches_sorted['log_gdp_diff'].notna().sum()
pop_ok = df_matches_sorted['log_pop_diff'].notna().sum()
print(f'Done. GDP coverage: {gdp_ok}/{total} | Population coverage: {pop_ok}/{total}')
df_matches_sorted[['team_a', 'team_b', 'year', 'log_gdp_diff', 'log_pop_diff']].head(10)

## 2. Feature Engineering

In [ ]:
# ── 2A. Rest Day Differential ──────────────────────────────────────────────
df = df_matches_sorted.copy()

last_match_date = {}
rest_a_list, rest_b_list = [], []

for _, row in df.iterrows():
    ta, tb, date = row['team_a'], row['team_b'], row['date']
    rest_a = (date - last_match_date[ta]).days if ta in last_match_date and pd.notna(date) else np.nan
    rest_b = (date - last_match_date[tb]).days if tb in last_match_date and pd.notna(date) else np.nan
    rest_a_list.append(rest_a)
    rest_b_list.append(rest_b)
    if pd.notna(date):
        last_match_date[ta] = date
        last_match_date[tb] = date

df['rest_a']    = rest_a_list
df['rest_b']    = rest_b_list
df['rest_diff'] = df['rest_a'] - df['rest_b']
print('Rest day differential computed.')

In [ ]:
# ── 2B. Rolling xG & Shooting Efficiency with Exponential Time-Decay ───────
# NOTE: xG is simulated from goals + noise until real StatsBomb/API-Football
# data is wired in. Replace df['xg_a'] / df['xg_b'] with actual xG columns.

np.random.seed(42)
df['xg_a'] = (df['goals_a'] + np.random.normal(0, 0.3, len(df))).clip(lower=0)
df['xg_b'] = (df['goals_b'] + np.random.normal(0, 0.3, len(df))).clip(lower=0)
df['sot_rate_a'] = (0.35 + np.random.normal(0, 0.08, len(df))).clip(0.1, 0.8)
df['sot_rate_b'] = (0.35 + np.random.normal(0, 0.08, len(df))).clip(0.1, 0.8)

LAMBDA = 0.1   # decay factor
WINDOW = 5     # last N matches

def decay_average(values, dates, lambda_=LAMBDA):
    if not values:
        return np.nan
    values = np.array(values)
    if len(values) == 1:
        return float(values[0])
    max_date = max(dates)
    weights = np.exp(-lambda_ * np.array([(max_date - d).days for d in dates]))
    return float(np.average(values, weights=weights))

team_history = {}
roll_xg_diff_list  = []
roll_sot_diff_list = []

for _, row in df.iterrows():
    ta, tb = row['team_a'], row['team_b']
    date = row['date']
    ea, eb = row['elo_a'], row['elo_b']

    def get_rolling(team, window=WINDOW):
        hist = team_history.get(team, [])[-window:]
        if not hist:
            return np.nan, np.nan
        xg_vals  = [h['xg_for']   for h in hist]
        sot_vals = [h['sot_rate'] for h in hist]
        elo_opp  = [h['elo_opp']  for h in hist]
        dates    = [h['date']     for h in hist]
        adj_xg = [x * (e / 1500) for x, e in zip(xg_vals, elo_opp)]
        return decay_average(adj_xg, dates), decay_average(sot_vals, dates)

    xg_a_roll,  sot_a_roll  = get_rolling(ta)
    xg_b_roll,  sot_b_roll  = get_rolling(tb)

    roll_xg_diff_list.append(
        (xg_a_roll - xg_b_roll) if not (np.isnan(xg_a_roll) or np.isnan(xg_b_roll)) else np.nan
    )
    roll_sot_diff_list.append(
        (sot_a_roll - sot_b_roll) if not (np.isnan(sot_a_roll) or np.isnan(sot_b_roll)) else np.nan
    )

    for team, xg_for, sot_rate, elo_opp in [
        (ta, row['xg_a'], row['sot_rate_a'], eb),
        (tb, row['xg_b'], row['sot_rate_b'], ea),
    ]:
        team_history.setdefault(team, []).append({
            'date': date, 'xg_for': xg_for, 'sot_rate': sot_rate, 'elo_opp': elo_opp
        })

df['roll_xg_diff']  = roll_xg_diff_list
df['roll_sot_diff'] = roll_sot_diff_list
print('Rolling xG and shooting efficiency features computed.')

In [ ]:
# ── 2C. Assemble Final Feature Matrix ───────────────────────────────────────

FEATURES = [
    'elo_diff',       # Elo rating differential
    'roll_xg_diff',   # Opponent-adjusted rolling xG differential
    'roll_sot_diff',  # Rolling shooting efficiency differential
    'log_gdp_diff',   # Log GDP per capita PPP differential (year-accurate)
    'log_pop_diff',   # Log population ratio (year-accurate)
    'rest_diff',      # Rest days differential
    'is_knockout',    # 1 = knockout stage, 0 = group stage
]

TARGET = 'outcome'  # 0=Team A Win, 1=Draw, 2=Team B Win

df_model = df[FEATURES + [TARGET, 'goals_a', 'goals_b', 'year']].copy()

for col in FEATURES:
    df_model[col].fillna(df_model[col].median(), inplace=True)

print(f'Feature matrix shape: {df_model.shape}')
print(f'\nOutcome distribution:')
print(df_model[TARGET].value_counts().rename({0: 'A Win', 1: 'Draw', 2: 'B Win'}))
df_model[FEATURES].describe()

## 3. Train/Test Split

In [ ]:
X = df_model[FEATURES].values
y = df_model[TARGET].values

# Temporal split: train on 2002–2014 (4 WCs), test on 2018 & 2022 (2 WCs)
train_mask = df_model['year'] <= 2014
test_mask  = df_model['year'] >= 2018

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Train set : {len(X_train)} matches (2002–2014)')
print(f'Test set  : {len(X_test)} matches (2018–2022)')

## 4. Model Training — Multiclass Classification (Win / Draw / Lose)

In [ ]:
# ── 4A. XGBoost Classifier (Primary) ────────────────────────────────────────

xgb_model = xgb.XGBClassifier(
    objective        = 'multi:softprob',
    num_class        = 3,
    n_estimators     = 300,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = 'mlogloss',
    random_state     = 42,
)

xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

xgb_probs = xgb_model.predict_proba(X_test)
xgb_preds = xgb_model.predict(X_test)

print(f'\nXGBoost Accuracy : {accuracy_score(y_test, xgb_preds):.4f}')
print(f'XGBoost Log-Loss : {log_loss(y_test, xgb_probs):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, xgb_preds, target_names=['A Win', 'Draw', 'B Win']))

In [ ]:
# ── 4B. LightGBM Classifier (Alternative) ───────────────────────────────────

lgb_model = lgb.LGBMClassifier(
    objective        = 'multiclass',
    num_class        = 3,
    n_estimators     = 300,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1,
)

lgb_model.fit(X_train, y_train)

lgb_probs = lgb_model.predict_proba(X_test)
lgb_preds = lgb_model.predict(X_test)

print(f'LightGBM Accuracy : {accuracy_score(y_test, lgb_preds):.4f}')
print(f'LightGBM Log-Loss : {log_loss(y_test, lgb_probs):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, lgb_preds, target_names=['A Win', 'Draw', 'B Win']))

In [ ]:
# ── 4C. Feature Importance Plot ─────────────────────────────────────────────

importances = xgb_model.feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_df['feature'], feat_df['importance'], color='steelblue')
plt.xlabel('XGBoost Feature Importance (Gain)')
plt.title('World Cup Model — Feature Importances')
plt.tight_layout()
plt.show()

## 5. Knockout Match Post-Processing (Draw Masking)

In [ ]:
def predict_match(team_a, team_b, is_knockout, model=xgb_model):
    """
    Predict outcome probabilities for a match.
    For knockout matches, draw prob is redistributed to Win/Loss proportionally.
    """
    ea = get_elo(team_a)
    eb = get_elo(team_b)

    def last_roll(team):
        hist = team_history.get(team, [])
        if not hist:
            return 0.0, 0.35
        return hist[-1]['xg_for'], hist[-1]['sot_rate']

    xg_a, sot_a = last_roll(team_a)
    xg_b, sot_b = last_roll(team_b)

    current_year = datetime.now().year
    log_gdp_diff, log_pop_diff = get_socioeconomic_diff(team_a, team_b, current_year)
    log_gdp_diff = log_gdp_diff if not np.isnan(log_gdp_diff) else 0.0
    log_pop_diff = log_pop_diff if not np.isnan(log_pop_diff) else 0.0

    fv = np.array([[
        ea - eb,
        xg_a - xg_b,
        sot_a - sot_b,
        log_gdp_diff,
        log_pop_diff,
        0.0,
        int(is_knockout)
    ]])

    probs = model.predict_proba(fv)[0]
    p_win, p_draw, p_loss = probs

    if is_knockout:
        p_win_adj  = p_win  / (p_win + p_loss)
        p_loss_adj = p_loss / (p_win + p_loss)
        return {'team_a': team_a, 'team_b': team_b,
                'P_win': round(p_win_adj, 4), 'P_draw': 0.0, 'P_loss': round(p_loss_adj, 4)}
    else:
        return {'team_a': team_a, 'team_b': team_b,
                'P_win': round(p_win, 4), 'P_draw': round(p_draw, 4), 'P_loss': round(p_loss, 4)}


print(predict_match('Brazil', 'Germany', is_knockout=True))
print(predict_match('France', 'Argentina', is_knockout=False))

## 6. Goal Prediction — Poisson Regression & Over/Under Classifier

In [ ]:
# ── 6A. Dual Poisson XGBoost Regressors ─────────────────────────────────────

y_goals_a = df_model['goals_a'].values
y_goals_b = df_model['goals_b'].values

poisson_a = XGBRegressor(objective='count:poisson', n_estimators=300,
                          max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)
poisson_b = XGBRegressor(objective='count:poisson', n_estimators=300,
                          max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)

poisson_a.fit(X_train, y_goals_a[train_mask])
poisson_b.fit(X_train, y_goals_b[train_mask])

lam_a_preds = poisson_a.predict(X_test).clip(min=0)
lam_b_preds = poisson_b.predict(X_test).clip(min=0)

print('Poisson Regressors trained.')
print(f'Mean predicted  lambda_A: {lam_a_preds.mean():.2f} | lambda_B: {lam_b_preds.mean():.2f}')
print(f'Actual mean goals A: {y_goals_a[test_mask].mean():.2f} | B: {y_goals_b[test_mask].mean():.2f}')

In [ ]:
# ── 6B. Scoreline Probability Matrix ────────────────────────────────────────

def scoreline_matrix(lambda_a, lambda_b, max_goals=6):
    matrix = np.zeros((max_goals + 1, max_goals + 1))
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            matrix[i, j] = poisson.pmf(i, lambda_a) * poisson.pmf(j, lambda_b)
    return pd.DataFrame(
        matrix,
        index=[f'A={i}' for i in range(max_goals + 1)],
        columns=[f'B={j}' for j in range(max_goals + 1)]
    )

lam_a, lam_b = 1.6, 1.1
score_mat = scoreline_matrix(lam_a, lam_b)

plt.figure(figsize=(8, 6))
sns.heatmap(score_mat, annot=True, fmt='.3f', cmap='YlOrRd')
plt.title(f'Scoreline Probability Matrix (lambda_A={lam_a}, lambda_B={lam_b})')
plt.xlabel('Team B Goals')
plt.ylabel('Team A Goals')
plt.tight_layout()
plt.show()

mat = score_mat.values
print(f'P(A Win)={np.tril(mat, k=-1).sum():.3f} | P(Draw)={np.trace(mat):.3f} | P(B Win)={np.triu(mat, k=1).sum():.3f}')

In [ ]:
# ── 6C. Over/Under 2.5 Binary Classifier ────────────────────────────────────

THRESHOLD = 2.5
y_ou = (df_model['goals_a'] + df_model['goals_b'] > THRESHOLD).astype(int).values

ou_model = xgb.XGBClassifier(
    objective        = 'binary:logistic',
    n_estimators     = 300,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = 'logloss',
    random_state     = 42,
)

ou_model.fit(X_train, y_ou[train_mask], eval_set=[(X_test, y_ou[test_mask])], verbose=False)

ou_preds = ou_model.predict(X_test)
ou_probs = ou_model.predict_proba(X_test)[:, 1]

print(f'Over/Under 2.5 Accuracy : {accuracy_score(y_ou[test_mask], ou_preds):.4f}')
print(f'Over/Under 2.5 Log-Loss : {log_loss(y_ou[test_mask], ou_probs):.4f}')
print(classification_report(y_ou[test_mask], ou_preds, target_names=['Under 2.5', 'Over 2.5']))

## 7. Full Prediction Wrapper

In [ ]:
def full_match_prediction(team_a, team_b, is_knockout=False, ou_threshold=2.5):
    """
    Unified prediction: outcome probs, expected goals, top scoreline, over/under.
    """
    ea = get_elo(team_a)
    eb = get_elo(team_b)

    def last_roll(team):
        hist = team_history.get(team, [])
        if not hist:
            return 0.0, 0.35
        return hist[-1]['xg_for'], hist[-1]['sot_rate']

    xg_a, sot_a = last_roll(team_a)
    xg_b, sot_b = last_roll(team_b)

    current_year = datetime.now().year
    log_gdp_diff, log_pop_diff = get_socioeconomic_diff(team_a, team_b, current_year)
    log_gdp_diff = log_gdp_diff if not np.isnan(log_gdp_diff) else 0.0
    log_pop_diff = log_pop_diff if not np.isnan(log_pop_diff) else 0.0

    fv = np.array([[
        ea - eb, xg_a - xg_b, sot_a - sot_b,
        log_gdp_diff, log_pop_diff, 0.0, int(is_knockout)
    ]])

    # Outcome
    probs = xgb_model.predict_proba(fv)[0]
    p_win, p_draw, p_loss = probs
    if is_knockout:
        p_win  = round(p_win  / (p_win + p_loss), 4)
        p_loss = round(p_loss / (p_win + p_loss + 1e-9), 4)
        p_draw = 0.0
    else:
        p_win, p_draw, p_loss = round(p_win, 4), round(p_draw, 4), round(p_loss, 4)

    # Expected goals
    lam_a = float(poisson_a.predict(fv).clip(min=0)[0])
    lam_b = float(poisson_b.predict(fv).clip(min=0)[0])

    # Top scoreline
    mat = scoreline_matrix(lam_a, lam_b)
    flat_max = mat.values.argmax()
    best_i, best_j = divmod(flat_max, mat.shape[1])

    # Over/Under
    ou_prob  = float(ou_model.predict_proba(fv)[0, 1])
    ou_label = f'Over {ou_threshold}' if ou_prob > 0.5 else f'Under {ou_threshold}'

    print('=' * 50)
    print(f'  {team_a}  vs  {team_b}  |  {"KNOCKOUT" if is_knockout else "GROUP STAGE"}')
    print('=' * 50)
    print(f'  Outcome   : Win={p_win:.1%}  Draw={p_draw:.1%}  Loss={p_loss:.1%}')
    print(f'  Exp Goals : {team_a} {lam_a:.2f}  |  {team_b} {lam_b:.2f}')
    print(f'  Top Score : {team_a} {best_i}-{best_j} {team_b}  (P={mat.values[best_i, best_j]:.3f})')
    print(f'  O/U {ou_threshold}   : {ou_label}  (P_over={ou_prob:.1%})')
    print('=' * 50)

    return {
        'P_win': p_win, 'P_draw': p_draw, 'P_loss': p_loss,
        'lambda_a': lam_a, 'lambda_b': lam_b,
        'best_scoreline': f'{best_i}-{best_j}',
        'ou_prediction': ou_label, 'ou_prob_over': ou_prob
    }


full_match_prediction('Brazil', 'Germany', is_knockout=True)
full_match_prediction('France', 'Argentina', is_knockout=False)

## Notes & Next Steps

| Area | Action |
|---|---|
| **Real xG data** | Replace simulated xG with StatsBomb Open Data or API-Football premium |
| **FIFA Rankings** | Add log-FIFA rank differential from historical Kaggle datasets |
| **Transfermarkt** | Merge squad market value differential from pre-scraped Kaggle datasets |
| **Hyperparameter tuning** | Use `optuna` or `GridSearchCV` on XGBoost/LightGBM |
| **Calibration** | Apply `CalibratedClassifierCV` to ensure reliable probabilities |
| **WB cache** | Save `_wb_cache` to disk to avoid re-fetching on each run |